In [1]:
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

/var/folders/b1/8s0sms7j4qq9c_gxxj46p5qw0000gn/T/ipykernel_7578/3777615979.py:1: DeprecationWarning: Importing display from IPython.core.display is deprecated since IPython 7.14, please import from IPython display
  from IPython.core.display import display, HTML


# Lab | Natural Language Processing
### SMS: SPAM or HAM

### Let's prepare the environment

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer

- Read Data for the Fraudulent Email Kaggle Challenge
- Reduce the training set to speead up development. 

In [4]:
## Read Data for the Fraudulent Email Kaggle Challenge
data = pd.read_csv("kg_train.csv",encoding='latin-1')

# Reduce the training set to speed up development. 
# Modify for final system
data = data.head(1000)
print(data.shape)
data.fillna("",inplace=True)

(1000, 2)


### Let's divide the training and test set into two partitions

In [7]:
# Your code

print(data.columns)
data.head()



Index(['text', 'label'], dtype='object')


,text,label
0,"DEAR SIR, STRICTLY A PRIVATE BUSINESS PROPOSAL...",1
1,Will do.,0
2,Nora--Cheryl has emailed dozens of memos about...,0
3,Dear Sir=2FMadam=2C I know that this proposal ...,1
4,fyi,0


In [9]:
X = data["text"]
y = data["label"]

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y 
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (800,)
Test shape: (200,)


## Data Preprocessing

In [11]:
import nltk
nltk.download("stopwords")

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/koldozablaetaaguirre/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [12]:
import string
from nltk.corpus import stopwords
print(string.punctuation)
print(stopwords.words("english")[100:110])
from nltk.stem.snowball import SnowballStemmer
snowball = SnowballStemmer('english')

!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~
['needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on']


## Now, we have to clean the html code removing words

- First we remove inline JavaScript/CSS
- Then we remove html comments. This has to be done before removing regular tags since comments can contain '>' characters
- Next we can remove the remaining tags

In [15]:
# Your code

import re
from bs4 import BeautifulSoup

def clean_html(text):

    soup = BeautifulSoup(text, "html.parser")
    
    for script in soup(["script", "style"]):
        script.decompose()
    
    # Eliminar comentarios HTML
    for element in soup(text=lambda text: isinstance(text, type(soup.comment))):
        element.extract()
    
    
    clean_text = soup.get_text(separator=" ")
    
    return clean_text

- Remove all the special characters
    
- Remove numbers
    
- Remove all single characters
 
- Remove single characters from the start

- Substitute multiple spaces with single space

- Remove prefixed 'b'

- Convert to Lowercase

In [16]:
# Your code

def clean_text_advanced(text):
    
    text = text.lower()
    
    # Quitar prefijo b'
    text = re.sub(r"\bb'", "", text)
    
    # Quitar números
    text = re.sub(r"\d+", "", text)
    
    # Quitar carácteres especiales
    text = re.sub(r"[^\w\s]", " ", text)
    
    # Quitar palabras de un solo carácter
    text = re.sub(r"\b[a-zA-Z]\b", "", text)
    
    # Quitar caracteres sueltos al inicio
    text = re.sub(r"^\s*[a-zA-Z]\s+", "", text)
    
    # Reemplazar múltiples espacios por uno solo
    text = re.sub(r"\s+", " ", text)
    
    return text.strip()

## Now let's work on removing stopwords
Remove the stopwords.

In [19]:
# Your code

stop_words = set(stopwords.words("english"))

def remove_stopwords(text):
    words = text.split()
    filtered_words = [word for word in words if word not in stop_words]
    return " ".join(filtered_words)

## Tame Your Text with Lemmatization
Break sentences into words, then use lemmatization to reduce them to their base form (e.g., "running" becomes "run"). See how this creates cleaner data for analysis!

In [22]:
# Your code

from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

def apply_lemmatization(text):
    words = text.split()
    lemmatized_words = [lemmatizer.lemmatize(word) for word in words]
    return " ".join(lemmatized_words)



## Bag Of Words
Let's get the 10 top words in ham and spam messages (**EXPLORATORY DATA ANALYSIS**)

In [25]:
# Your code

ham_text = data[data["label"] == 0]["clean_text"]
spam_text = data[data["label"] == 1]["clean_text"]

from sklearn.feature_extraction.text import CountVectorizer
import numpy as np

def get_top_words(text_series, n=10):
    vectorizer = CountVectorizer()
    X = vectorizer.fit_transform(text_series)
    
    word_counts = np.sum(X.toarray(), axis=0)
    words = vectorizer.get_feature_names_out()
    
    word_freq = list(zip(words, word_counts))
    word_freq = sorted(word_freq, key=lambda x: x[1], reverse=True)
    
    return word_freq[:n]


print("Top 10 HAM words:")
print(get_top_words(ham_text))

print("\nTop 10 SPAM words:")
print(get_top_words(spam_text))



Top 10 HAM words:
[('the', 1774), ('to', 1064), ('and', 834), ('of', 793), ('in', 616), ('that', 414), ('is', 385), ('for', 374), ('on', 328), ('you', 311)]

Top 10 SPAM words:
[('the', 7076), ('to', 5601), ('of', 4993), ('and', 3995), ('in', 3298), ('you', 3245), ('this', 2685), ('my', 2154), ('your', 2082), ('for', 2042)]


## Extra features

In [26]:
# We add to the original dataframe two additional indicators (money symbols and suspicious words).
money_simbol_list = "|".join(["euro","dollar","pound","€",r"\$"])
suspicious_words = "|".join(["free","cheap","sex","money","account","bank","fund","transfer","transaction","win","deposit","password"])

data_train['money_mark'] = data_train['preprocessed_text'].str.contains(money_simbol_list)*1
data_train['suspicious_words'] = data_train['preprocessed_text'].str.contains(suspicious_words)*1
data_train['text_len'] = data_train['preprocessed_text'].apply(lambda x: len(x)) 

data_val['money_mark'] = data_val['preprocessed_text'].str.contains(money_simbol_list)*1
data_val['suspicious_words'] = data_val['preprocessed_text'].str.contains(suspicious_words)*1
data_val['text_len'] = data_val['preprocessed_text'].apply(lambda x: len(x)) 

data_train.head()

NameError: name 'data_train' is not defined

## How would work the Bag of Words with Count Vectorizer concept?

In [27]:
# Your code

from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd

list = [
    "I love machine learning",
    "Machine learning is amazing",
    "I love coding"
]

vectorizer = CountVectorizer()

X = vectorizer.fit_transform(list)

print("Vocabulary:")
print(vectorizer.get_feature_names_out())

bow_df = pd.DataFrame(X.toarray(), columns=vectorizer.get_feature_names_out())
print("\nBag of Words Matrix:")
print(bow_df)

Vocabulary:
['amazing' 'coding' 'is' 'learning' 'love' 'machine']

Bag of Words Matrix:
   amazing  coding  is  learning  love  machine
0        0       0   0         1     1        1
1        1       0   1         1     0        1
2        0       1   0         0     1        0


## TF-IDF

- Load the vectorizer

- Vectorize all dataset

- print the shape of the vetorized dataset

In [ ]:
# Your code

from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=5000,
    stop_words="english"
)

X_tfidf = tfidf.fit_transform(data["clean_text"])


print("TF-IDF shape:", X_tfidf.shape)

TF-IDF shape: (1000, 5000)


## And the Train a Classifier?

In [ ]:
# Your code

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

tfidf = TfidfVectorizer(max_features=5000, stop_words="english")

X_train_tfidf = tfidf.fit_transform(X_train)  
X_test_tfidf  = tfidf.transform(X_test)

#  Entrenar clasificador
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train_tfidf, y_train)

# Predicciones y evaluación
y_pred = clf.predict(X_test_tfidf)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Accuracy: 0.975

Confusion Matrix:
 [[112   0]
 [  5  83]]

Classification Report:
               precision    recall  f1-score   support

           0       0.96      1.00      0.98       112
           1       1.00      0.94      0.97        88

    accuracy                           0.97       200
   macro avg       0.98      0.97      0.97       200
weighted avg       0.98      0.97      0.97       200



### Extra Task - Implement a SPAM/HAM classifier

https://www.kaggle.com/t/b384e34013d54d238490103bc3c360ce

The classifier can not be changed!!! It must be the MultinimialNB with default parameters!

Your task is to **find the most relevant features**.

For example, you can test the following options and check which of them performs better:
- Using "Bag of Words" only
- Using "TF-IDF" only
- Bag of Words + extra flags (money_mark, suspicious_words, text_len)
- TF-IDF + extra flags


You can work with teams of two persons (recommended).

In [ ]:
# Your code